# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This simulator generates synthetic FHIR Observation & Patient streams via PySpark memory streams into the Bronze layer.

In [ ]:
import os
import uuid
import json
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, udf
from pyspark.sql.types import StringType

os.environ['SPARK_VERSION'] = '3.3'

spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.apache.iceberg:iceberg-aws-bundle:1.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("SparkSession initialized for Streaming Simulator.")

In [ ]:
# Define UDFs
def generate_uuid():
    return str(uuid.uuid4())

uuid_udf = udf(generate_uuid, StringType())

def generate_fhir_payload(id_val):
    # Simple mock FHIR payload
    payload = {
        "resourceType": "Patient",
        "id": f"pat_{id_val}",
        "active": True,
        "name": [{"use": "official", "family": "Doe", "given": ["John", "A."]}]
    }
    return json.dumps(payload)

fhir_udf = udf(generate_fhir_payload, StringType())

# Create a rate stream to simulate incoming Kafka messages
df_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 2) \
    .load()

# Transform into Bronze schema
df_bronze_stream = df_stream \
    .withColumn("payload_id", uuid_udf()) \
    .withColumn("source_system_id", lit("MOCK_FHIR_STREAM")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("raw_payload_json", fhir_udf(df_stream.value)) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

# Write stream to Iceberg table
# For local simulation, we can just write to a memory table or local iceberg path
query = df_bronze_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .option("checkpointLocation", "/tmp/ehdip_checkpoints/day02_fhir_stream") \
    .toTable("local.bronze_db.raw_payloads")

print("Started streaming query...")

# Let it run for a short time for simulation purposes
time.sleep(15)
query.stop()
print("Stopped streaming query.")

In [ ]:
# Verify data landed in Bronze
spark.sql("SELECT * FROM local.bronze_db.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 5").show(truncate=False)